# 02 - RDD (Partie 2, Personne 1)
Chargement du JSONL en RDD, nettoyage, puis revenus par ville, durée moyenne et top 5 des stations de départ. Résultats comparés à la référence indépendante de la Personne 2 (`data/validate_data.py`).

In [1]:
import sys
sys.path.append('/home/jovyan/work/pipeline')
from pedali_spark import get_spark
import rdd_analysis

spark = get_spark('pedali-02-rdd')
res = rdd_analysis.run(spark)

In [2]:
print('Lignes brutes      :', res['lignes_brutes'])
print('Trajets terminés   :', res['trajets_termines'])
print('Revenus par ville   :', res['revenus_par_ville'])
print('Durée moyenne (min):', res['duree_moyenne_min'])
print('Top 5 stations     :', res['top5_stations'])

Lignes brutes      : 20100
Trajets terminés   : 17673
Revenus par ville   : {'Casablanca': 157812.0, 'Marrakech': 144175.5, 'Rabat': 163402.5}
Durée moyenne (min): 17.56
Top 5 stations     : [('S01', 2815), ('S11', 2142), ('S10', 1938), ('S05', 1512), ('S07', 1485)]


## Lignée (lineage) du RDD `done`
Chaque `reduceByKey` introduit un shuffle (nouveau stage dans le DAG).

In [3]:
print(res['lineage'])

(2) PythonRDD[6] at RDD at PythonRDD.scala:53 [Memory Serialized 1x Replicated]
 |       CachedPartitions: 2; MemorySize: 943.3 KiB; DiskSize: 0.0 B
 |  MapPartitionsRDD[5] at mapPartitions at PythonRDD.scala:160 [Memory Serialized 1x Replicated]
 |  ShuffledRDD[4] at partitionBy at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]
 +-(2) PairwiseRDD[3] at reduceByKey at /home/jovyan/work/pipeline/rdd_analysis.py:28 [Memory Serialized 1x Replicated]
    |  PythonRDD[2] at reduceByKey at /home/jovyan/work/pipeline/rdd_analysis.py:28 [Memory Serialized 1x Replicated]
    |  file:///home/jovyan/work/data/trips_history.jsonl MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]
    |  file:///home/jovyan/work/data/trips_history.jsonl HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]


## Contrôle avec la référence Python pur (Personne 2)

In [4]:
import subprocess
print(subprocess.run(['python','/home/jovyan/work/data/validate_data.py'],capture_output=True,text=True).stdout)

20100 lignes, 12 stations, 100 trip_id dupliques
  [!!] doublons (trip_id repete): 200
  [!!] user_id manquant: 100
  [!!] station de depart inconnue: 100
  [!!] duree negative: 91
  [OK] trajet termine sans end_time: 0

Reference apres nettoyage (17673 trajets termines)
Revenus par ville : {'Casablanca': 157812.0, 'Marrakech': 144175.5, 'Rabat': 163402.5}
Duree moyenne (min): 17.56
Top 5 stations de depart: [('S01', 2815), ('S11', 2142), ('S10', 1938), ('S05', 1512), ('S07', 1485)]



In [5]:
spark.stop()